In [ ]:
# More imports
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import torch.nn as nn
import torch
from torch.optim import AdamW

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(os.path.join(path, 'Q3_data.csv'))

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
df_clean = df.fillna(0).copy()
df_clean.head()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_clean.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

print('\n\nHead:')
df_clean.head()

In [ ]:
# Task 4: Write your code here:
#### No categorical values exist, all types are either float64 or int64

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, 'Target')

In [ ]:
### Refering to the chart above, we can see that the target imbalanced

In [ ]:
feature_cols = df_clean.drop('Target', axis=1).columns

In [ ]:
# Task 1: Write your code here:
X = df_clean[feature_cols]
y = df_clean['Target']

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Task 2,3,4,5: Write your code here:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

# CatBoosClassifier model
class CatBoosClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim):
    super(CatBoosClassifier, self).__init__()

    # First linear layer: input features -> hidden layer
    self.layer1 = nn.Linear(input_dim, hidden_dim)

    # Second linear layer: hidden layer -> hidden layer
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)

    # Output layer: hidden layer -> single continuous value
    self.layer3 = nn.Linear(hidden_dim, 1)

    # ReLU activation for non-linearity in hidden layers
    self.relu = nn.ReLU()

  # Defines how input data flows through the network
  def forward(self, x):
    # First hidden layer
    a1 = self.relu(self.layer1(x))

    # Second hidden layer
    a2 = self.relu(self.layer2(a1))

    # Output layer (regression output)
    output = self.layer3(a2)

    return output

#Training Loop
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # Set the model to training mode
  model.train()

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # Move batch to the selected device
    X_batch = X_batch.to(device)              # shape: (batch_size, num_features)
    y_batch = y_batch.view(-1, 1).to(device) # shape: (batch_size, 1)

    # Forward pass (continuous output)
    outputs = model(X_batch)                  # shape: (batch_size, 1)
    loss = criterion(outputs, y_batch)

    # Backward pass & optimization
    optimizer.zero_grad()   # Clear previous gradients
    loss.backward()         # Compute gradients
    optimizer.step()        # Update model parameters

    running_loss += loss.item()

  # Average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

input_dim = X_train.shape[1]
hidden_dim = 32
learning_rate = 0.001
model = CatBoosClassifier(input_dim, hidden_dim).to(device)
num_epochs = 10
optimizer = AdamW(model.parameters(), learning_rate)
criterion = nn.MSELoss()
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Run training
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  # Train one epoch
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # Validate
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  if (epoch + 1) % 5 == 0:
    print(
      f'Epoch [{epoch+1}/{num_epochs}], '
      f'Train Loss: {train_loss:.4f}, '
      f'Val Loss: {val_loss:.4f}'
    )

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
#not enough time

In [ ]:
# Task Bonus: Write your code here: